# 09 · Thesis and publication figures

Simple figures made for print. Every figure:
* is sized for a journal page: **one column = 90 mm**, **two columns = 190 mm** wide;
* uses 7-8 pt text at that printed size and has panel labels **(a), (b), (c)**, units on every axis and no title (the caption says what it shows);
* gives each model the same colour everywhere;
* is saved as **PDF** (vector, for the journal) and **PNG at 600 dpi** (for Word).

Figures that show borehole locations are saved in `private/` (keep them out of the thesis until the data owners agree). Borehole names are hidden: **BH-A, BH-B, BH-C**.

| Fig. | Content |
|---|---|
| 1 | Study area and boreholes *(private)* |
| 2 | Clay share by depth |
| 3 | blockCV: (a) range per depth slice, (b) folds, (c) purge example *(private)* |
| 4 | Model comparison: Macro-F1 and balanced accuracy |
| 5 | Macro-F1 vs distance to the nearest training borehole |
| 6 | Confusion matrix and mean spatial ROC curve (final model and XGB) |
| 7 | SHAP importance, permutation importance and SHAP values (final model and XGB) |
| 8 | Elevation: observed clay share vs the models' elevation effect |
| 9 | External tests |
| 10 | Soil columns: measured vs predicted, three test boreholes |
| 11 | 3D map: P(Clay) at four depths with reliability zones *(private)* |
| S1, S2 | Block-size sensitivity; purge |

**Run time:** 1-2 minutes.

In [ ]:
import json, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "settings.toml").exists())))
import helpers as H

S = H.load_settings()
T0 = time.time()
print("QUICK TEST RUN - not thesis results" if S["run"]["quick"] else "FULL RUN", "| V3 folder:", S["v3_root"])
import joblib
import geopandas as gpd
from matplotlib.colors import LinearSegmentedColormap, ListedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Rectangle
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve

OUT, PRIV = H.out_dir(S, "09_figures"), H.out_dir(S, "09_figures", private=True)
cells = pd.read_csv(H.processed_dir(S) / "cells.csv")
dev = cells[cells.set == "development"]
preds = pd.read_csv(H.out_dir(S, "03_spatial_cv", private=True) / "outer_predictions.csv")
summ = pd.read_csv(H.out_dir(S, "04_results") / "metrics_fold_mean_sd.csv")
final = json.loads((H.out_dir(S, "05_final_model") / "final_model.json").read_text())
FINAL = final["model"]
MAIN = H.block_sizes(S)[0]
main = preds[(preds.block_m == MAIN) & (preds.purge_m == 0)]
MODELS = [m for m in S["cv"]["models"] if m in set(main.model)]
EXTRA = "XGB"                                   # also shown in Figs. 6, 7, 8 and 10

# ---- style -------------------------------------------------------------------------------
MM = 1 / 25.4
W1, W2 = 90 * MM, 190 * MM
COL = {"RF": "#17365C", "XGB": "#17918C", "MLP": "#E5B74A", "GWRF": "#6E0032", "GWXGB": "#A0004E"}
NICE = {"RF": "RF", "XGB": "XGBoost", "MLP": "MLP", "GWRF": "GWRF", "GWXGB": "GWXGBoost"}
CLAY, SAND, GREY = "#5B8C3A", "#F2CF63", "#9E9E9E"
CMAP_P = LinearSegmentedColormap.from_list("sand_clay", [SAND, "#FFFFFF", CLAY])
CMAP_V = LinearSegmentedColormap.from_list("navy_magenta", ["#17365C", "#D9A6C1", "#A0004E"])
plt.rcParams.update({"font.family": "sans-serif", "font.size": 8, "axes.labelsize": 8, "axes.titlesize": 8,
                     "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7, "axes.linewidth": 0.6,
                     "xtick.major.width": 0.6, "ytick.major.width": 0.6, "lines.linewidth": 1.0,
                     "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False,
                     "pdf.fonttype": 42, "savefig.dpi": 600, "savefig.bbox": "tight", "figure.dpi": 150})

def label(ax, s, x=-0.14, y=1.04):
    ax.text(x, y, f"({s})", transform=ax.transAxes, fontsize=9, fontweight="bold", va="bottom")

def save(fig, name, private=False):
    folder = PRIV if private else OUT
    for ext in ("pdf", "png"):
        fig.savefig(folder / f"{name}.{ext}")
    print("saved", folder / f"{name}.pdf")

mark = lambda m: m + (" *" if m == FINAL else "")
km = lambda v, _: f"{v / 1000:.0f}"

## Fig. 1 · Study area (private)

In [ ]:
poly = H.polygon(S, "downtown_shp")
bh = cells.drop_duplicates("BH_Code").set_index("BH_Code")
new_records, _ = H.load_new_external(S)
new_bh = new_records.groupby("BH_Code")[["UTM_Easting_m", "UTM_Northing_m"]].first()
fig, ax = plt.subplots(figsize=(W1, W1 * 0.8))
gpd.GeoSeries([poly]).boundary.plot(ax=ax, color="black", linewidth=0.7)
for s_, m_, c_, lab in (("development", "o", "#17365C", "Development"), ("external", "^", "#A0004E", "External holdout")):
    b = bh[bh.set == s_]
    ax.scatter(b.UTM_Easting_m, b.UTM_Northing_m, s=8, marker=m_, color=c_, label=f"{lab} ({len(b)})", zorder=3, linewidth=0)
ax.scatter(new_bh.UTM_Easting_m, new_bh.UTM_Northing_m, s=10, marker="s", facecolor="none", edgecolor="#17918C",
           linewidth=0.7, label=f"New external ({len(new_bh)})", zorder=3)
x0, y0, x1, y1 = poly.bounds
ax.plot([x1 - 2500, x1 - 500], [y0 + 400] * 2, color="black", linewidth=1.5)
ax.text(x1 - 1500, y0 + 550, "2 km", ha="center", fontsize=7)
ax.annotate("N", xy=(x0 + 400, y1 - 200), xytext=(x0 + 400, y1 - 1300), ha="center", fontsize=8,
            arrowprops={"arrowstyle": "-|>", "linewidth": 0.8, "color": "black"})
ax.xaxis.set_major_formatter(km); ax.yaxis.set_major_formatter(km)
ax.set_xlabel("Easting (km, UTM 48N)"); ax.set_ylabel("Northing (km)"); ax.set_aspect("equal")
ax.legend(loc="lower left", bbox_to_anchor=(0, 1.0), ncol=2, handletextpad=0.2, columnspacing=0.8)
save(fig, "fig01_study_area", private=True)

## Fig. 2 · Clay share by depth

In [ ]:
prof = pd.read_csv(H.out_dir(S, "01_data") / "clay_share_by_depth.csv", index_col=0)
fig, ax = plt.subplots(figsize=(W1 * 0.6, W1 * 0.8))
ax.plot(prof["measured cells"], prof.index + 0.25, "-o", color="#17365C", markersize=2.5, label="Measured cells")
ax.plot(prof["all cells"], prof.index + 0.25, "--", color="#A0004E", label="Measured + filled")
ax.set_ylim(15, 0); ax.set_xlim(0, 1)
ax.set_xlabel("Clay share"); ax.set_ylabel("Depth (m)")
ax.legend(loc="lower right")
save(fig, "fig02_clay_share_by_depth")

## Fig. 3 · blockCV block size and folds (private)

In [ ]:
dec = json.loads((S["v3_root"] / "data" / "folds" / "block_size_decision.json").read_text())
ranges = pd.Series(dec["ranges_m"])
folds = H.load_folds(S, MAIN)
blocks_file = H.v3_outputs(S, "04_blockcv_folds", "restricted") / f"blocks_{MAIN}m.gpkg"
FOLD_COL = ["#2C6FBB", "#F28E2B", "#59A14F", "#C0000B", "#7B5CA8"]
b_dev = dev.drop_duplicates("BH_Code").set_index("BH_Code")[["UTM_Easting_m", "UTM_Northing_m"]]
b_dev["fold"] = folds.reindex(b_dev.index).astype(int)
fig, axes = plt.subplots(1, 3, figsize=(W2, W2 * 0.36), gridspec_kw={"width_ratios": [0.8, 1, 1]})
ax = axes[0]
ax.barh(range(len(ranges)), ranges.values, color="#17365C", height=0.7)
ax.axvline(dec["block_size_m"], color="#A0004E", linewidth=1)
ax.text(dec["block_size_m"] + 4, -0.6, f"median = {dec['block_size_m']} m", color="#A0004E", fontsize=7, va="bottom")
ax.set_yticks(range(len(ranges)), [r.replace(" m", "") for r in ranges.index]); ax.invert_yaxis()
ax.set_xlabel("blockCV range (m)"); ax.set_ylabel("Depth slice (m)")
label(ax, "a", x=-0.45)
for i, ax in enumerate(axes[1:]):
    if blocks_file.exists():
        blk = gpd.read_file(blocks_file)
        occ = blk[blk.intersects(gpd.GeoSeries(gpd.points_from_xy(b_dev.UTM_Easting_m, b_dev.UTM_Northing_m), crs=blk.crs).union_all())]
        for k in range(1, 6):
            occ[occ.folds == k].plot(ax=ax, color=FOLD_COL[k - 1], alpha=0.25, linewidth=0)
    for k in range(1, 6):
        b = b_dev[b_dev.fold == k]
        ax.scatter(b.UTM_Easting_m, b.UTM_Northing_m, s=5, color=FOLD_COL[k - 1], label=f"Fold {k}", zorder=3, linewidth=0)
    ax.xaxis.set_major_formatter(km); ax.yaxis.set_major_formatter(km); ax.set_aspect("equal")
    ax.set_xlabel("Easting (km)")
axes[1].set_ylabel("Northing (km)"); axes[2].set_ylabel("")
axes[1].legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=3, fontsize=6, handletextpad=0.1, columnspacing=0.6)
test, train = b_dev[b_dev.fold == 1], b_dev[b_dev.fold != 1]
d = H.nearest_distance(train.values[:, :2], test.values[:, :2])
removed = train[d < MAIN]
for x, y in test.values[:, :2]:
    axes[2].add_patch(plt.Circle((x, y), MAIN, fill=False, color="#555555", linewidth=0.4, zorder=2))
axes[2].scatter(removed.UTM_Easting_m, removed.UTM_Northing_m, s=12, marker="x", color="black", linewidth=0.7, zorder=4)
axes[2].legend([Line2D([], [], color="#555555", lw=0.6), Line2D([], [], marker="x", ls="", color="black")],
               [f"{MAIN} m around fold-1 test boreholes", f"removed training boreholes ({len(removed)})"],
               loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=1, fontsize=6)
label(axes[1], "b"); label(axes[2], "c")
fig.tight_layout(w_pad=1.0)
save(fig, "fig03_blockcv", private=True)

## Fig. 4 · Model comparison

In [ ]:
s = summ[(summ.block_m == MAIN) & (summ.purge_m == 0) & summ.model.isin(MODELS)].set_index("model")
order = list(s.sort_values("macro_f1_mean", ascending=False).index)
fig, axes = plt.subplots(1, 2, figsize=(W2 * 0.75, W1 * 0.62), sharey=True)
for ax, k, lab, pl in zip(axes, ["macro_f1", "balanced_accuracy"], ["Macro-F1", "Balanced accuracy"], "ab"):
    v, e = s.loc[order, f"{k}_mean"], s.loc[order, f"{k}_sd"].fillna(0)
    ax.bar(range(len(order)), v, yerr=e, color=[COL[m] for m in order], width=0.7, capsize=2,
           error_kw={"elinewidth": 0.6, "capthick": 0.6})
    for i, (vi, ei) in enumerate(zip(v, e)):
        ax.text(i, vi + ei + 0.015, f"{vi:.3f}", ha="center", fontsize=6.5)
    ax.set_xticks(range(len(order)), [NICE[m] + (" *" if m == FINAL else "") for m in order], rotation=30, ha="right")
    ax.set_ylabel(lab); ax.set_ylim(0, 1); label(ax, pl)
fig.tight_layout()
save(fig, "fig04_model_comparison")
display(s.loc[order, ["macro_f1_mean", "macro_f1_sd", "balanced_accuracy_mean", "balanced_accuracy_sd"]].round(3))

## Fig. 5 · Macro-F1 vs distance to the nearest training borehole

In [ ]:
dist = pd.read_csv(H.out_dir(S, "04_results") / "metrics_by_distance_to_training.csv")
dist = dist[dist.model.isin(MODELS) & (dist.block_m == MAIN) & (dist.purge_m == 0)]
bins = [b for b in ["0-50 m", "50-100 m", "100-200 m", "200-300 m", "300-500 m", "500-1000 m", ">1000 m"] if b in set(dist.distance_bin)]
fig, ax = plt.subplots(figsize=(W1, W1 * 0.7))
for m in MODELS:
    t = dist[dist.model == m].set_index("distance_bin").reindex(bins)
    ax.plot(range(len(bins)), t.macro_f1, "-o", color=COL[m], markersize=3, label=NICE[m],
            linewidth=1.6 if m == FINAL else 0.9)
n = dist.groupby("distance_bin").n.max().reindex(bins)
ax.set_xticks(range(len(bins)), [f"{b.replace(' m', '')}\n(n={int(v)})" for b, v in zip(bins, n)], fontsize=6)
ax.set_xlabel("Distance to the nearest training borehole (m)"); ax.set_ylabel("Macro-F1")
ax.set_ylim(0, 1); ax.legend(ncol=2, loc="lower left")
save(fig, "fig05_score_by_distance")

## Fig. 6 · Confusion matrix and mean spatial ROC curve

In [ ]:
show = [FINAL] + ([EXTRA] if EXTRA != FINAL and EXTRA in MODELS else [])
fig, axes = plt.subplots(len(show), 2, figsize=(W2 * 0.7, W1 * 0.62 * len(show)), squeeze=False,
                         gridspec_kw={"width_ratios": [1, 1.15]})
grid_fpr = np.linspace(0, 1, 201)
for row, m in enumerate(show):
    f = main[main.model == m]
    cm = confusion_matrix(f.y, f.y_pred, labels=[0, 1])
    pct = cm / cm.sum(axis=1, keepdims=True)
    ax = axes[row, 0]
    ax.imshow(pct, cmap=LinearSegmentedColormap.from_list("w", ["#FFFFFF", COL[m]]), vmin=0, vmax=1)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i, j]}\n({pct[i, j]:.0%})", ha="center", va="center", fontsize=7,
                    color="white" if pct[i, j] > 0.55 else "black")
    ax.set_xticks([0, 1], ["Sand", "Clay"]); ax.set_yticks([0, 1], ["Sand", "Clay"])
    ax.set_xlabel("Predicted"); ax.set_ylabel(f"{NICE[m]}{' *' if m == FINAL else ''}\nObserved")
    for sp in ax.spines.values():
        sp.set_visible(True)
    ax = axes[row, 1]
    tprs, aucs = [], []
    for k, g in f.groupby("outer_fold"):
        if g.y.nunique() == 2:
            fpr, tpr, _ = roc_curve(g.y, g.p_clay)
            tprs.append(np.interp(grid_fpr, fpr, tpr)); tprs[-1][0] = 0
            aucs.append(roc_auc_score(g.y, g.p_clay))
    mt_, sd_ = np.mean(tprs, axis=0), np.std(tprs, axis=0)
    mt_[-1] = 1
    ax.fill_between(grid_fpr, np.clip(mt_ - sd_, 0, 1), np.clip(mt_ + sd_, 0, 1), color=COL[m], alpha=0.15, linewidth=0)
    ax.plot(grid_fpr, mt_, color=COL[m], label=f"Mean AUC = {np.mean(aucs):.3f} ± {np.std(aucs, ddof=1) if len(aucs) > 1 else 0:.3f}")
    ax.plot([0, 1], [0, 1], ":", color=GREY, linewidth=0.8)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect("equal")
    ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate"); ax.legend(loc="lower right")
    label(axes[row, 0], "ab"[0] if row == 0 else "c", x=-0.45); label(ax, "b" if row == 0 else "d", x=-0.3)
fig.tight_layout()
save(fig, "fig06_confusion_roc")

## Fig. 7 · Explainable AI (held-out cells)

In [ ]:
shap_df = pd.read_csv(H.out_dir(S, "06_xai", private=True) / "shap_values_by_cell.csv")
perm = pd.read_csv(H.out_dir(S, "06_xai") / "permutation_importance_summary.csv")
FEAT = {"UTM_Easting_m": "Easting", "UTM_Northing_m": "Northing", "Top_Elevation_m": "Elevation"}
fig, axes = plt.subplots(len(show), 3, figsize=(W2, W1 * 0.55 * len(show)), squeeze=False,
                         gridspec_kw={"width_ratios": [0.8, 1, 1.3]})
letters = iter("abcdef")
for row, m in enumerate(show):
    sv = shap_df[shap_df.model == m]
    imp = pd.Series({v: sv[f"shap_{v}"].abs().mean() for v in FEAT.values()}).sort_values()
    ax = axes[row, 0]
    ax.barh(imp.index, imp.values, color=COL[m], height=0.6)
    ax.set_xlabel("Mean |SHAP| on P(Clay)"); ax.set_ylabel(NICE[m] + (" *" if m == FINAL else ""))
    label(ax, next(letters), x=-0.55)
    p = perm[(perm.model == m) & (perm.cells == "all cells")].set_index("input")
    names = ["Easting", "Northing", "Elevation", "Location (Easting + Northing)", "Elevation (depth trend)"]
    p = p.reindex([n_ for n_ in names if n_ in p.index])
    ax = axes[row, 1]
    ax.barh(range(len(p)), p["mean"], xerr=p["std"], color=[COL[m] if k_ == "single" else GREY for k_ in p.kind],
            height=0.6, error_kw={"elinewidth": 0.6})
    ax.set_yticks(range(len(p)), [n_.replace(" (Easting + Northing)", " (E + N)").replace(" (depth trend)", " (group)") for n_ in p.index])
    ax.invert_yaxis(); ax.axvline(0, color="black", linewidth=0.5)
    ax.set_xlabel("Drop in balanced accuracy"); label(ax, next(letters), x=-0.55)
    ax = axes[row, 2]
    for i, (c, v) in enumerate(FEAT.items()):
        x = sv[f"shap_{v}"].values
        val = sv[c].values
        col = (val - val.min()) / (np.ptp(val) or 1)
        jit = np.random.default_rng(i).normal(0, 0.09, len(x))
        sc = ax.scatter(x, i + jit, c=col, cmap=CMAP_V, s=2, alpha=0.7, linewidth=0, vmin=0, vmax=1)
    ax.set_yticks(range(3), list(FEAT.values())); ax.axvline(0, color="black", linewidth=0.5)
    ax.set_xlabel("SHAP value (+ = towards Clay)"); label(ax, next(letters), x=-0.25)
    cb = fig.colorbar(sc, ax=ax, ticks=[0, 1], fraction=0.05, pad=0.02)
    cb.ax.set_yticklabels(["low", "high"]); cb.set_label("Input value", fontsize=7)
fig.tight_layout()
save(fig, "fig07_xai")

## Fig. 8 · Elevation: observed clay share and the models' elevation effect

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(W1 * 1.3, W1 * 0.75), sharey=True)
b = pd.cut(dev.Top_Elevation_m, 14)
obs = dev.groupby(b, observed=True).agg(z=("Top_Elevation_m", "mean"), clay=("y", "mean"))
axes[0].plot(obs.clay, obs.z, "-o", color="black", markersize=2.5)
axes[0].set_xlabel("Observed clay share"); axes[0].set_ylabel("Top elevation of cell (m)"); axes[0].set_xlim(0, 1)
for m in show:
    d = shap_df[shap_df.model == m]
    t = d.groupby(pd.cut(d.Top_Elevation_m, 14), observed=True).agg(z=("Top_Elevation_m", "mean"), s=("shap_Elevation", "mean"))
    axes[1].plot(t.s, t.z, "-o", color=COL[m], markersize=2.5, label=NICE[m] + (" *" if m == FINAL else ""))
axes[1].axvline(0, color="black", linewidth=0.5)
axes[1].set_xlabel("Mean SHAP of elevation"); axes[1].legend(loc="lower right")
label(axes[0], "a", x=-0.3); label(axes[1], "b", x=-0.1)
fig.tight_layout()
save(fig, "fig08_elevation_effect")

## Fig. 9 · External tests

In [ ]:
ext = pd.read_csv(H.out_dir(S, "07_external_tests") / "external_tests.csv")
ext = ext[(ext.group == "all") & (ext.cells == "all cells")].set_index("test")
metrics = [("macro_f1", "Macro-F1"), ("balanced_accuracy", "Balanced acc."), ("roc_auc", "ROC-AUC"), ("brier", "Brier")]
tests = list(ext.index)
colors = ["#17365C", "#17918C", "#E5B74A"]
fig, ax = plt.subplots(figsize=(W1 * 1.3, W1 * 0.62))
w = 0.8 / len(tests)
for i, t in enumerate(tests):
    v = ext.loc[t, [k for k, _ in metrics]].values.astype(float)
    ax.bar(np.arange(len(metrics)) + (i - (len(tests) - 1) / 2) * w, v, width=w * 0.95, color=colors[i % 3],
           label=f"{t} (n = {int(ext.loc[t, 'n'])} cells)")
ax.set_xticks(range(len(metrics)), [l_ for _, l_ in metrics]); ax.set_ylim(0, 1)
ax.set_ylabel("Score (Brier: lower is better)"); ax.legend(loc="upper left", bbox_to_anchor=(0, 1.25), fontsize=6.5)
save(fig, "fig09_external_tests")
display(ext[[k for k, _ in metrics] + ["n"]].round(3))

## Fig. 10 · Soil columns: measured vs predicted (three test boreholes)
The boreholes are chosen by a rule, not by eye, from the final model's outer-fold results:
* **BH-A**: within 100 m of a training borehole, with the accuracy closest to the median of that group;
* **BH-B**: the borehole with the accuracy closest to the median of all test boreholes;
* **BH-C**: more than 300 m from a training borehole, with the accuracy closest to the median of that group.

The measured column is drawn like a boring log: cells without a sample take the class of the nearest sample (the boundary between two classes is at the midpoint). Below the end of the borehole the column is hatched. The predicted columns are the held-out outer-fold models' predictions for every 0.5 m cell.

In [ ]:
f = main[main.model == FINAL].assign(correct=lambda d: (d.y == d.y_pred).astype(float))
fb = f.groupby("BH_Code").agg(acc=("correct", "mean"), dist=("dist_nearest_train_m", "first"), fold=("outer_fold", "first"))
def closest_to_median(d):
    return None if d.empty else (d.acc - d.acc.median()).abs().sort_values(kind="stable").index[0]
picked = []
for crit in (fb.dist <= 100, fb.index.notna(), fb.dist > 300):
    c = closest_to_median(fb[crit & ~fb.index.isin(picked)])
    if c is not None:
        picked.append(c)
names = dict(zip(picked, ["BH-A", "BH-B", "BH-C"]))
records = H.load_downtown(S)
bottom = records.groupby("BH_Code").Depth_To_m.max()
tops = np.arange(0, 15, 0.5)
col_models = [m for m in ["GWRF", "GWXGB", EXTRA] if m in MODELS]
MODELS_DIR = H.out_dir(S, "03_spatial_cv", private=True) / "models"

def measured_column(code):
    c = cells[cells.BH_Code == code].set_index("cell").y
    cls = np.full(30, np.nan)
    for j in range(30):
        if tops[j] >= bottom[code]:
            continue
        nearest = c.index[np.argmin(np.abs(c.index.values - j))]      # nearest labelled cell; ties -> the upper one
        cls[j] = c[nearest]
    return cls

def predicted_column(code, m):
    r = cells[cells.BH_Code == code].iloc[0]
    X = np.column_stack([np.full(30, r.UTM_Easting_m), np.full(30, r.UTM_Northing_m), r.Ground_Elevation_m - tops])
    k = int(fb.loc[code, "fold"])
    model = joblib.load(MODELS_DIR / f"A_{MAIN}m_p0_{m}_fold{k}.joblib")
    thr = main[(main.model == m) & (main.outer_fold == k)].threshold.iloc[0]
    return (model.predict_proba(X)[:, 1] >= thr).astype(float)

fig, axes = plt.subplots(1, len(picked), figsize=(W2 * 0.85, W1 * 0.9), sharey=True)
axes = np.atleast_1d(axes)
for ax, code, pl in zip(axes, picked, "abc"):
    columns = [("Measured", measured_column(code))] + [(NICE[m] + (" *" if m == FINAL else ""), predicted_column(code, m)) for m in col_models]
    for i, (name, cls) in enumerate(columns):
        for j, v in enumerate(cls):
            if np.isnan(v):
                ax.add_patch(Rectangle((i + 0.1, tops[j]), 0.8, 0.5, facecolor="white", edgecolor=GREY, hatch="////", linewidth=0))
            else:
                ax.add_patch(Rectangle((i + 0.1, tops[j]), 0.8, 0.5, facecolor=CLAY if v == 1 else SAND, linewidth=0))
        ax.add_patch(Rectangle((i + 0.1, 0), 0.8, 15, fill=False, edgecolor="black", linewidth=0.6))
    ax.set_xlim(0, len(columns)); ax.set_ylim(15, 0)
    ax.set_xticks(np.arange(len(columns)) + 0.5, [n_ for n_, _ in columns], rotation=35, ha="right")
    ax.spines["bottom"].set_visible(False); ax.tick_params(axis="x", length=0)
    ax.set_title(f"{names[code]}\n{fb.loc[code, 'dist']:.0f} m from training BH\naccuracy {fb.loc[code, 'acc']:.2f} ({NICE[FINAL]})", fontsize=7)
    label(ax, pl, x=-0.12, y=1.2)
axes[0].set_ylabel("Depth (m)")
fig.legend([Patch(color=CLAY), Patch(color=SAND), Patch(facecolor="white", edgecolor=GREY, hatch="////")],
           ["Clay (CL, CH)", "Sand (SC, SM, SP)", "Below end of borehole"], loc="lower center", ncol=3,
           bbox_to_anchor=(0.5, -0.08))
fig.tight_layout()
save(fig, "fig10_soil_columns")
print({names[c]: {"accuracy": round(fb.loc[c, "acc"], 3), "distance_m": round(fb.loc[c, "dist"])} for c in picked})

## Fig. 11 · 3D map: P(Clay) at four depths (private)

In [ ]:
spacing = S["quick"]["map_spacing_m"] if S["run"]["quick"] else S["map"]["spacing_m"]
grid = pd.read_csv(H.out_dir(S, "08_map", private=True) / f"siemreap_3d_points_120bh_{int(spacing)}m.csv")
fig, axes = plt.subplots(2, 2, figsize=(W2 * 0.8, W2 * 0.62), sharex=True, sharey=True)
for ax, depth, pl in zip(axes.ravel(), [1.0, 5.0, 10.0, 14.0], "abcd"):
    g = grid[np.isclose(grid.depth_top_m, depth)]
    sc = ax.scatter(g.X, g.Y, c=g.p_clay, cmap=CMAP_P, vmin=0, vmax=1, s=0.4 if spacing <= 50 else 4, marker="s", linewidth=0)
    z3 = g[g.reliability_zone == 3]
    ax.scatter(z3.X, z3.Y, s=0.05 if spacing <= 50 else 0.5, color="black", alpha=0.25, linewidth=0)
    ax.set_aspect("equal"); ax.xaxis.set_major_formatter(km); ax.yaxis.set_major_formatter(km)
    ax.text(0.02, 0.96, f"{depth:g}-{depth + 0.5:g} m", transform=ax.transAxes, fontsize=7, va="top")
    label(ax, pl, x=-0.12, y=1.0)
for ax in axes[1]:
    ax.set_xlabel("Easting (km)")
for ax in axes[:, 0]:
    ax.set_ylabel("Northing (km)")
cb = fig.colorbar(sc, ax=axes, fraction=0.025, pad=0.02)
cb.set_label("P(Clay)")
fig.legend([Line2D([], [], marker="o", ls="", color="black", alpha=0.4, markersize=2)],
           ["grey dots: more than 300 m from a borehole (reliability zone 3)"], loc="lower center", bbox_to_anchor=(0.45, -0.02))
save(fig, "fig11_map_depths", private=True)

## Supplementary figures S1 and S2 · Block-size sensitivity and purge

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(W2 * 0.7, W1 * 0.55), sharey=True)
for m in S["cv"]["sensitivity_models"]:
    t = summ[(summ.model == m) & (summ.purge_m == 0)].sort_values("block_m")
    axes[0].errorbar(t.block_m, t.macro_f1_mean, yerr=t.macro_f1_sd, fmt="-o", color=COL[m], markersize=3, capsize=2,
                     elinewidth=0.6, label=NICE[m])
    t = summ[(summ.model == m) & (summ.block_m == MAIN)].sort_values("purge_m")
    axes[1].errorbar(t.purge_m, t.macro_f1_mean, yerr=t.macro_f1_sd, fmt="-o", color=COL[m], markersize=3, capsize=2,
                     elinewidth=0.6, label=NICE[m])
axes[0].set_xlabel("Block size (m)"); axes[0].set_ylabel("Macro-F1 (mean ± SD)")
axes[1].set_xlabel(f"Purge distance (m), {MAIN} m blocks"); axes[0].set_ylim(0, 1); axes[0].legend()
label(axes[0], "a", x=-0.25); label(axes[1], "b", x=-0.1)
fig.tight_layout()
save(fig, "figS1_block_size_and_purge")
print(f"Finished in {time.time() - T0:.0f} s.")